# Trabajo Práctico 2 - Grupo 02

### Ensamble — BETO v3.1 + XGBoost v6

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen

## 1. Instalación de dependencias necesarias con Keggle Notebook y corrección de errores

In [1]:
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 75.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# ------------------------------------------------------------
# 0. Fix del circular import de datasets (bug en Kaggle)
# ------------------------------------------------------------
import importlib.abc, importlib.util, sys

class _NilLoader(importlib.abc.Loader):
    def create_module(self, spec):
        return None
    def exec_module(self, module):
        module.Dataset = type("Dataset", (), {})

class _BlockDatasets:
    def find_spec(self, fullname, path=None, target=None):
        if fullname == "datasets" or fullname.startswith("datasets."):
            return importlib.util.spec_from_loader(
                fullname, _NilLoader(), is_package=(fullname == "datasets")
            )
        return None

sys.meta_path.insert(0, _BlockDatasets())
print("datasets circular import fix applied")

# ------------------------------------------------------------
# Configuración
# ------------------------------------------------------------
sys.path.insert(0, "/kaggle/input/datasets/tiagoandrcaldern/common-files")

NOMBRE_ENTREGA = "ensamble_beto_v3.1_xgb_v6"

BETO_MODEL_PATH = "/kaggle/input/datasets/tiagoandrcaldern/beto-v3-1-model"

# Pesos para soft voting [beto, xgboost]
PESOS = [0.60, 0.40]

print(f"Entrega: {NOMBRE_ENTREGA}")
print(f"Pesos: {PESOS}")

datasets circular import fix applied
Entrega: ensamble_beto_v3.1_xgb_v6
Pesos: [0.6, 0.4]


## 2. Imports y configuración

In [3]:
import os
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score

from common.data_utils import get_split, make_bow, SEED
from common.preprocessing import clean_minimal, clean_classical
from common.evaluation import evaluate

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LENGTH = 128
BATCH_SIZE = 64

print(f"Device: {DEVICE}")
np.random.seed(SEED)

Device: cuda


## 3. Carga de datos

In [4]:
DATA = "/kaggle/input/datasets/tiagoandrcaldern/data-files-ensamble-xg-beto"

train_df      = pd.read_csv(f"{DATA}/train.csv")
train_eda_df  = pd.read_csv(f"{DATA}/train_augmented_eda_balanced.csv")
test_df       = pd.read_csv(f"{DATA}/test.csv")

# Split de validación sobre train original
X_train_raw, X_val_raw, y_train, y_val = get_split(train_df)

# Preprocesamiento para BETO: clean_minimal
X_val_minimal  = np.array([clean_minimal(t) for t in X_val_raw])
X_test_minimal = np.array([clean_minimal(t) for t in test_df["text"].values])

# Preprocesamiento para XGBoost: clean_classical
X_val_classical  = np.array([clean_classical(t) for t in X_val_raw])
X_test_classical = np.array([clean_classical(t) for t in test_df["text"].values])

print(f"Val: {len(X_val_minimal):,} | Test: {len(X_test_minimal):,}")

Val: 10,200 | Test: 8,500


## 4. Funciones de inferencia

In [5]:
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding="max_length",
            max_length=max_length, return_tensors="pt",
        )
    def __len__(self):
        return len(self.encodings["input_ids"])
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()}


def get_transformer_probs(model_dir, texts):
    """Carga un transformer guardado y devuelve matriz (n, 3) de probabilidades."""
    print(f"  Cargando {model_dir} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
    model     = AutoModelForSequenceClassification.from_pretrained(model_dir, local_files_only=True)
    model     = model.to(DEVICE)
    model.eval()

    dataset    = TextDataset(texts, tokenizer, MAX_LENGTH)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    all_probs = []
    with torch.no_grad():
        for batch in dataloader:
            batch  = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = model(**batch).logits
            probs  = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)

    del model
    torch.cuda.empty_cache()
    return np.vstack(all_probs)

print("Funciones cargadas.")

Funciones cargadas.


## 5. XGBoost — entrenamiento (modelo clásico)

In [6]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

print("Entrenando XGBoost v6 ...")

# Preprocesar dataset EDA completo
X_eda_raw = np.array([clean_classical(t) for t in train_eda_df["text"].values])
y_eda     = train_eda_df["label"].values

# Split estratificado
from sklearn.model_selection import train_test_split
X_eda_train, _, y_eda_train, _ = train_test_split(
    X_eda_raw, y_eda, test_size=0.2, stratify=y_eda, random_state=SEED
)

# Bow vectorizer
bow = make_bow()
X_train_bow = bow.fit_transform(X_eda_train)
X_val_bow   = bow.transform(X_val_classical)
X_test_bow  = bow.transform(X_test_classical)

print(f"BoW features: {X_train_bow.shape[1]:,}")

# XGBoost v6 hyperparams
xgb = XGBClassifier(
    n_estimators=700,
    max_depth=7,
    learning_rate=0.2,
    colsample_bytree=1.0,
    min_child_weight=1,
    subsample=0.8,
    reg_lambda=2,
    reg_alpha=0,
    eval_metric="mlogloss",
    random_state=SEED,
    n_jobs=-1,
)

sample_weights = compute_sample_weight("balanced", y_eda_train)
xgb.fit(X_train_bow, y_eda_train, sample_weight=sample_weights)

# Evaluación local
y_pred_xgb = xgb.predict(X_val_bow)
f1_xgb_val = f1_score(y_val, y_pred_xgb, average="macro")
print(f"XGBoost F1-macro (val): {f1_xgb_val:.4f}")

# Probabilidades
xgb_val_probs  = xgb.predict_proba(X_val_bow)
xgb_test_probs = xgb.predict_proba(X_test_bow)
print("XGBoost listo.")

Entrenando XGBoost v6 ...
BoW features: 31,611
XGBoost F1-macro (val): 0.7862
XGBoost listo.


## 6. BETO v3.1 — inferencia (transformer)

In [7]:
print("BETO v3.1 — generando predicciones ...")

beto_val_probs  = get_transformer_probs(BETO_MODEL_PATH, X_val_minimal)
beto_test_probs = get_transformer_probs(BETO_MODEL_PATH, X_test_minimal)

y_pred_beto = np.argmax(beto_val_probs, axis=1)
f1_beto_val = f1_score(y_val, y_pred_beto, average="macro")
print(f"BETO F1-macro (val): {f1_beto_val:.4f}")
print("BETO listo.")

BETO v3.1 — generando predicciones ...
  Cargando /kaggle/input/datasets/tiagoandrcaldern/beto-v3-1-model ...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Cargando /kaggle/input/datasets/tiagoandrcaldern/beto-v3-1-model ...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BETO F1-macro (val): 0.7744
BETO listo.


## 7. Ensamble por soft voting ponderado

In [8]:
# Juntar probabilidades de ambos modelos
val_probs_list  = [beto_val_probs, xgb_val_probs]
test_probs_list = [beto_test_probs, xgb_test_probs]

# Promedio ponderado
val_probs_ensemble  = sum(w * p for w, p in zip(PESOS, val_probs_list))
test_probs_ensemble = sum(w * p for w, p in zip(PESOS, test_probs_list))

# Predicción final: clase con mayor probabilidad
y_pred_val  = np.argmax(val_probs_ensemble,  axis=1)
y_pred_test = np.argmax(test_probs_ensemble, axis=1)

# Evaluación del ensamble
evaluate(NOMBRE_ENTREGA, y_val, y_pred_val,
         hyperparams={
             "modelos": ["BETO_v3.1", "XGBoost_v6"],
             "pesos": PESOS,
         })


=== ensamble_beto_v3.1_xgb_v6 ===
Hiperparámetros: {'modelos': ['BETO_v3.1', 'XGBoost_v6'], 'pesos': [0.6, 0.4]}

F1-macro:  0.8069
Precision: 0.8099
Recall:    0.8047
Accuracy:  0.8424

              precision    recall  f1-score   support

    negativa     0.8685    0.8691    0.8688      4080
      neutra     0.6642    0.6167    0.6396      2040
    positiva     0.8970    0.9284    0.9124      4080

    accuracy                         0.8424     10200
   macro avg     0.8099    0.8047    0.8069     10200
weighted avg     0.8390    0.8424    0.8404     10200

Matriz de confusión (filas=real, cols=predicho):
          negativa  neutra  positiva
negativa      3546     433       101
neutra         448    1258       334
positiva        89     203      3788


## 8. Comparación individual vs ensamble

In [9]:
nombres = ["BETO_v3.1", "XGBoost_v6"]
probs   = val_probs_list

print("F1-macro en validación:")
print(f"  Ensamble (pesos {PESOS}): {f1_score(y_val, y_pred_val, average='macro'):.4f}")
for i, (p, w) in enumerate(zip(probs, PESOS)):
    preds_i = np.argmax(p, axis=1)
    f1_i    = f1_score(y_val, preds_i, average="macro")
    print(f"  {nombres[i]} (peso={w}): {f1_i:.4f}")

F1-macro en validación:
  Ensamble (pesos [0.6, 0.4]): 0.8069
  BETO_v3.1 (peso=0.6): 0.7744
  XGBoost_v6 (peso=0.4): 0.7862


## 9. Submission a Kaggle

In [10]:
Path("submissions").mkdir(exist_ok=True)

sub = pd.DataFrame({"id": test_df["id"].values, "label": y_pred_test.astype(int)})
sub.to_csv(f"submissions/submission_{NOMBRE_ENTREGA}.csv", index=False)

dist = sub["label"].value_counts(normalize=True).sort_index()
print(f"Guardado: submissions/submission_{NOMBRE_ENTREGA}.csv  ({len(sub)} predicciones)")
print(f"Distribucion: {', '.join(f'clase {k}: {v:.1%}' for k, v in dist.items())}")

Guardado: submissions/submission_ensamble_beto_v3.1_xgb_v6.csv  (8500 predicciones)
Distribucion: clase 0: 42.3%, clase 1: 17.5%, clase 2: 40.2%
